In [ ]:
import pandas as pd
import json

from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR,NuSVR

import pandas as pd
from helpers.modeling import (
    identify_column_types,
    create_preprocessor,
    evaluate_model,
)

from tqdm import tqdm


with open('results.json', 'r') as f:
    results = json.load(f)

In [59]:
df = pd.read_csv("../Datasets/processed/UHPC_dataset/semantic_recoding_features_50_with_publications.csv")
df.head()
df['paper_reference'].nunique()
df['paper_reference'].value_counts().describe()

count    165.000000
mean      12.563636
std       15.086454
min        1.000000
25%        4.000000
50%        8.000000
75%       16.000000
max      112.000000
Name: count, dtype: float64

In [60]:
def run_pipeline(model_cls, model_key, kernel_kwargs=None):
    params = results["best_params"]["best_params_publications_included"][model_key]
    pipeline = Pipeline([('preprocessor', preprocessor),
                          ('model', model_cls(**(kernel_kwargs or {})))])
    pipeline.set_params(**params)
    pipeline.fit(X_train, y_train)

    train_metrics = evaluate_model(y_train, pipeline.predict(X_train))
    test_metrics = evaluate_model(y_test, pipeline.predict(X_test))
    return pipeline, train_metrics, test_metrics

In [61]:
X = df.drop(columns=["cs_28d", "paper_reference"])
y = df["cs_28d"]

pub_col = "paper_reference" 

numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)

preprocessor = create_preprocessor(numerical_cols, one_hot_columns, k_fold_columns, 
                                   handle_unknown='ignore') 


In [62]:
total_rows = 0
for pub, group in df.groupby('paper_reference'):
    print(f"{pub}: {len(group)} rows")
    total_rows += len(group)
    
print(total_rows)

Ref-1-data: 16 rows
Ref-10-Research: 7 rows
Ref-100-Research: 8 rows
Ref-101-Research: 28 rows
Ref-102-Research: 16 rows
Ref-103-Research: 10 rows
Ref-104-Research: 8 rows
Ref-105-Research: 7 rows
Ref-106-Research: 8 rows
Ref-107-Research: 16 rows
Ref-108-Research: 3 rows
Ref-109-Research: 2 rows
Ref-11-Research : 4 rows
Ref-110-Research: 18 rows
Ref-111-Research: 2 rows
Ref-113-Research: 5 rows
Ref-114-Research: 5 rows
Ref-115-Research: 5 rows
Ref-116-Research: 36 rows
Ref-117-Research: 5 rows
Ref-118-Research: 5 rows
Ref-119-Research: 1 rows
Ref-12-Research: 4 rows
Ref-120-Research: 4 rows
Ref-121-Research: 80 rows
Ref-122-Research: 18 rows
Ref-123-Research: 11 rows
Ref-124-Research: 10 rows
Ref-125-Research: 16 rows
Ref-126-Research: 5 rows
Ref-127-Research: 3 rows
Ref-128-Research: 3 rows
Ref-129-Research: 1 rows
Ref-13-Research: 27 rows
Ref-130-Research: 6 rows
Ref-132-Research: 12 rows
Ref-133-Research: 2 rows
Ref-134-Research: 20 rows
Ref-135-Research: 43 rows
Ref-136-Research: 

In [63]:
model_configs = [
    (KNeighborsRegressor, 'knn', None),
    (SVR, 'svr', {'kernel': 'rbf'}),
    (NuSVR, 'nusvr', {'kernel': 'rbf'}),
]

groups = list(df.groupby(pub_col))
threshold = 5
lopo_results = []
lopo_predictions = []  
total_rows = 0

for pub_id, group_df in tqdm(groups, total=len(groups)):
    if len(group_df) < threshold:   #threshold
        continue
    total_rows += len(group_df)

    train_mask = df[pub_col] != pub_id
    test_mask  = df[pub_col] == pub_id

    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]

    for model_cls, model_key, kwargs in model_configs:
        pipeline, train_metrics, test_metrics = run_pipeline(model_cls, model_key, kwargs)
        test_metrics.update({'publication': pub_id, 'model': model_key, 'train_RMSE': train_metrics['RMSE']})
        lopo_results.append(test_metrics)

        y_pred = pipeline.predict(X_test)
        for true_val, pred_val, idx in zip(y_test.values, y_pred, y_test.index):
            lopo_predictions.append({'index': idx, 'publication': pub_id, 'model': model_key,
                                       'y_true': true_val, 'y_pred': pred_val})

lopo_df = pd.DataFrame(lopo_results)
lopo_pred_df = pd.DataFrame(lopo_predictions)
print(f"total rows after thresholding ({threshold}) : {total_rows}")

  0%|          | 0/165 [00:00<?, ?it/s]

100%|██████████| 165/165 [04:41<00:00,  1.71s/it]

total rows after thresholding (5) : 1947


In [64]:
print(f"unique values after threshold  {threshold}: {lopo_df['publication'].nunique()}")


unique values after threshold  5: 123


In [65]:
# Pooled LOPO summary:
# Every row in the dataset was held out exactly once (by the model that never
# saw its publication during training). This pools ALL those held-out
# predictions together (across all 165 eligible publications) and computes
# ONE overall RMSE/MAE/R2/Correlation/Mean_Residual per model - as if it were
# a single big "unseen publication" test set.

pooled_summary = []
for model_key, group in lopo_pred_df.groupby('model'):
    m = evaluate_model(group['y_true'], group['y_pred'])
    m['model'] = model_key
    m['N_total'] = len(group)
    pooled_summary.append(m)

pooled_lopo_df = pd.DataFrame(pooled_summary)
pooled_lopo_df

,RMSE,MAE,R2,Correlation,Mean_Residual,N,model,N_total
0,23.773496,17.940708,0.571403,0.762057,-0.763231,1947,knn,1947
1,26.470966,19.867248,0.468624,0.696432,0.269476,1947,nusvr,1947
2,26.891111,20.364134,0.451622,0.688420,0.419095,1947,svr,1947


In [66]:
lopo_pred_df['residual'] = lopo_pred_df['y_true'] - lopo_pred_df['y_pred']

residual_pivot = lopo_pred_df.pivot(index='index', columns='model', values='residual')
residual_pivot['mean_abs_residual'] = residual_pivot[['knn', 'svr', 'nusvr']].abs().mean(axis=1)
residual_pivot['same_direction'] = residual_pivot[['knn', 'svr', 'nusvr']].apply(
    lambda r: (r > 0).all() or (r < 0).all(), axis=1
)

worst_rows = residual_pivot.sort_values('mean_abs_residual', ascending=False).head(10)
worst_rows

model,knn,nusvr,svr,mean_abs_residual,same_direction
index,,,,,
1579,113.358687,87.509044,86.534104,95.800612,True
1580,90.746615,82.098156,88.119906,86.988226,True
1851,83.951064,86.413765,81.827033,84.063954,True
547,31.732461,105.818344,110.002990,82.517932,True
548,29.734868,103.569482,108.613337,80.639229,True
103,62.160048,83.406255,84.233945,76.600083,True
549,24.736889,97.464736,102.051376,74.751000,True
551,41.381688,87.397374,90.092317,72.957126,True
527,31.070710,91.550571,95.974215,72.865166,True


In [67]:
worst_idx = worst_rows.head(10).index 

df.loc[worst_idx].T

index,1579,1580,1851,547,548,103,549,551,527,1848
cement,991.74,1061.95,740.06,850.0,850.0,824.0,850.0,850.0,850.0,777.42
cement_type,Unknown,Unknown,white_cement,OPC_53,OPC_53,OPC_52.5,OPC_53,OPC_53,OPC_53,white_cement
silica_fume,247.93,265.49,185.02,260.0,260.0,0.0,260.0,260.0,260.0,194.5
fly_ash,0.0,0.0,179.68,0.0,0.0,0.0,0.0,0.0,0.0,188.57
fly_ash_type,not_applicable,not_applicable,unknown_type,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,unknown_type
limestone_powder,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
quartz_powder,0.0,0.0,0.0,212.0,212.0,0.0,212.0,212.0,212.0,0.0
glass_powder,247.93,265.49,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
rice_husk_ash,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
metakaolin,0.0,0.0,0.0,0.0,0.0,204.0,0.0,0.0,0.0,0.0


In [68]:
lopo_df.to_csv("results/lopo_results.csv", index=False)

In [69]:
summary = lopo_df.groupby('model')[['RMSE', 'MAE', 'R2', 'Correlation', 'Mean_Residual']].agg(['mean', 'median', 'std'])
summary

RMSE                              MAE                        \
            mean     median        std       mean     median        std   
model                                                                     
knn    20.887911  18.377611  11.151101  17.787194  15.693547  10.292921   
nusvr  21.901311  18.704314  12.305978  19.055446  15.787558  11.691448   
svr    22.038263  19.230893  12.213197  19.218409  16.698940  11.525584   

             R2                     Correlation                      \
           mean   median        std        mean    median       std   
model                                                                 
knn   -7.181572 -0.42171  34.271435    0.386135  0.554292  0.540685   
nusvr -6.051514 -0.77038  18.348120    0.478822  0.672114  0.487682   
svr   -6.039673 -0.55900  18.274659    0.491274  0.677115  0.492912   

      Mean_Residual                       
               mean    median        std  
model                                     
knn       -0.881471 -0.540705  17.357502  
nusvr     -1.273171 -2.973312  19.904253  
svr       -1.539899 -2.599933  19.931866

In [70]:
per_pub = lopo_df[['publication', 'model', 'N', 'RMSE', 'MAE', 'R2', 'Correlation', 'Mean_Residual']].sort_values(['model', 'publication'])
per_pub.to_csv("results/lopo_per_publication.csv", index=False)

In [ ]:
#worst publivations
worst = lopo_df.sort_values('R2').groupby('model').head(10)[['model', 'publication', 'N', 'RMSE', 'R2', 'Correlation', 'Mean_Residual']]
worst

,model,publication,N,RMSE,R2,Correlation,Mean_Residual
321,knn,Ref-75-Research,5,20.544520,-356.729010,0.532811,20.504829
323,nusvr,Ref-75-Research,5,14.431795,-175.523645,0.166203,14.311017
322,svr,Ref-75-Research,5,13.898565,-162.720132,0.212512,13.593586
210,knn,Ref-37-Research,7,36.313152,-122.526515,-0.598053,-35.693657
211,svr,Ref-37-Research,7,33.408774,-103.557058,-0.435416,-32.754319
212,nusvr,Ref-37-Research,7,29.501030,-80.527970,-0.201363,-29.112940
36,knn,Ref-114-Research,5,31.643315,-40.776608,-0.717109,23.319592
240,knn,Ref-49-Research,6,25.418035,-36.514120,0.479129,25.150656
71,nusvr,Ref-13-Research,27,25.124980,-32.599970,-0.010983,-23.142908
292,svr,Ref-64-Research,8,23.282731,-32.487913,0.596344,-21.258287


In [72]:
lopo_df['direction'] = lopo_df['Mean_Residual'].apply(lambda x: 'under-predicted' if x > 0 else 'over-predicted')
direction_summary = lopo_df.groupby(['model', 'direction']).size().unstack(fill_value=0)
direction_summary

direction,over-predicted,under-predicted
model,,
knn,63,60
nusvr,68,55
svr,64,59
